# 02 · Preprocessing, Rule-based Extraction & Ethics (PII)

**Objective.** Build the shared text-cleaning foundation, demonstrate tokenization / normalization / lemmatization, and add a **PII-redaction** step (privacy/ethics).

**Basic techniques covered:** *Text Preprocessing Pipeline*, *Rule-based Information Extraction (regex)*.
**Advanced technique covered:** *LLM with privacy/fairness/ethics considerations* (PII redaction).

>  Logic lives in `src/`; these notebooks orchestrate, experiment, visualise and analyse (good-practice separation, ULO6). Run the notebooks in order `01 -> 07` after completing setup in `README.md`.

In [1]:
# --- bootstrap: make `import config` and `from src import ...` work from anywhere ---
import sys, os
from pathlib import Path
p = Path.cwd()
ROOT = next((c for c in [p, *p.parents] if (c/'config.py').exists() and (c/'src').exists()), p)
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print('project root:', ROOT)
import config  # make config.* available to later cells

project root: C:\Users\lenovo\Desktop\ANLP_8420_GROUPC\Assignment3_GroupC\Codes


## 1. Two cleaning paths
Classical models want aggressive normalisation (lowercased, punctuation stripped); the LLM and NER want the original case/punctuation. We keep both, on purpose.

In [2]:
from src.preprocessing import clean_for_classical, clean_for_llm
msg = "My ORDER #48213 hasn't arrived!! Visit https://acme.io/track now."
print('classical:', clean_for_classical(msg))
print('llm      :', clean_for_llm(msg))

classical: my order #48213 hasn't arrived visit now
llm      : My ORDER #48213 hasn't arrived!! Visit now.


## 2. Tokenization, Normalization, and Lemmatization (spaCy)

After cleaning the text, we apply spaCy's linguistic pipeline to break messages into individual tokens and generate normalized forms of each word.

The output below shows the original token, its lowercase version, and its lemma. For example, *returning* is reduced to *return*, allowing different word forms to be treated consistently during analysis.

This step is particularly useful for traditional NLP techniques such as TF-IDF, sentiment analysis, and text classification, where reducing vocabulary variation often improves performance.

In [3]:
import spacy
nlp = spacy.load(config.SPACY_MODEL)
doc = nlp('The customers were returning their damaged orders quickly.')
import pandas as pd
pd.DataFrame([{'token': t.text, 'lemma': t.lemma_, 'lower': t.lower_,
               'is_stop': t.is_stop} for t in doc])

C:\Users\lenovo\anaconda3\envs\comp8420\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


,token,lemma,lower,is_stop
0,The,the,the,True
1,customers,customer,customers,False
2,were,be,were,True
3,returning,return,returning,False
4,their,their,their,True
5,damaged,damage,damaged,False
6,orders,order,orders,False
7,quickly,quickly,quickly,False
8,.,.,.,False


## 3. PII redaction (privacy / ethics)
Cheap, ethically important and explicitly rewarded. The same regex spans double as **silver labels** for NER evaluation in notebook 03/07.

In [4]:
from src.preprocessing import redact_pii
sample = 'Hi, I am Alex. Order #48213, email alex@acme.io, phone 0412 345 678, card 4111 1111 1111 1111.'
redacted, spans = redact_pii(sample)
print(redacted)
import pandas as pd
pd.DataFrame([{'type': s.label, 'value': s.value} for s in spans])

Hi, I am Alex. <ORDER_NUMBER>, email <EMAIL>, phone <PHONE>, card <CREDIT_CARD>.


,type,value
0,ORDER_NUMBER,Order #48213
1,EMAIL,alex@acme.io
2,PHONE,0412 345 678
3,CREDIT_CARD,4111 1111 1111 1111


## Ethics Discussion

Customer enquiries often contain sensitive information such as email addresses, phone numbers, order IDs, and payment details. To reduce privacy risks, these fields are automatically redacted before being passed to the LLM.

The example above shows how personal information can be replaced with placeholder tags while preserving the overall meaning of the message. This approach reduces the chance of exposing customer data and supports more responsible use of external language models.

To improve fairness during model training, class weighting is also applied to reduce the impact of class imbalance. This helps the classifier pay more attention to less frequent enquiry categories rather than favouring only the majority classes.

## 4. Bitext `{{placeholder}}` handling
Bitext encodes gold entities like `{{Order Number}}`. We can treat them as free silver NER labels, or fill them with realistic values for demos.

In [5]:
from src.preprocessing import extract_placeholder_entities, fill_placeholders, strip_placeholders
raw = 'Hi {{Person Name}}, your {{Order Number}} ships to {{Delivery Address}}.'
print('entities :', extract_placeholder_entities(raw))
print('filled   :', fill_placeholders(raw))
print('stripped :', strip_placeholders(raw))

entities : [{'text': 'Person Name', 'label': 'PERSON_NAME', 'start': 3, 'end': 14}, {'text': 'Order Number', 'label': 'ORDER_NUMBER', 'start': 21, 'end': 33}, {'text': 'Delivery Address', 'label': 'DELIVERY_ADDRESS', 'start': 43, 'end': 59}]
filled   : Hi Alex Morgan, your 48213 ships to 12 Baker Street, London.
stripped : Hi Person Name, your Order Number ships to Delivery Address.


## Summary

In this notebook, a reusable preprocessing pipeline was developed for the customer service system.

The pipeline supports:
- Text cleaning for both traditional NLP models and LLM workflows
- Tokenization, normalization, and lemmatization using spaCy
- PII detection and redaction for privacy protection
- Extraction of Bitext placeholders as silver-label entities for later NER evaluation

The outputs generated here are used throughout the remaining notebooks, providing a consistent preprocessing stage for classification, sentiment analysis, NER, retrieval, and response generation tasks.

**Next:** Notebook 03 develops the core NLP models, including intent classification, sentiment analysis, POS tagging, named entity recognition, and TF-IDF-based retrieval.